# Cleaning2
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [3]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


## Operazioni
- Eliminare i parametri con troppe poche righe
- Eliminare soffetti con solo 1 visita
- nuovi metadati (cofattori e fattori)

In [6]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned1'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned2'
create_new_support_file(support_file, support_file_path, new_name=new_name, rename=False)
dataCleaner = DataCleaner(support_file_path=new_name+'.xlsx')
new_support_file = pd.read_excel(new_name+'.xlsx')

In [7]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    # Eliminare i parametri con troppe poche righe
    df_cleaned, file_code, temp_support_file = dataCleaner.remove_param_few_subjects(df_new, file_name, prefix='cleaned/single_file')
    # --> funzione che trova i soggetti che hanno solo una visita quindi elimina quelle righe
    
    
    if 'ADSP_PHC_BIOMARKER' not in file_name:         # file solo con 1 visita, quindi da unire per ampliare il dataset ma non da usare da solo
        # Eliminare soggetti con solo 1 visita
        df_cleaned= dataCleaner.remove_sub_1visit(df_cleaned)
        # --> funzione che trova i parametri identificati da eliminare  ==> eliminare le colonne dal df
    
    final_df, bool_var = dataCleaner.classes_to_dummies(df_cleaned, col_list=['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']) #, 'DX'])

    # Nuovi metadati (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_02')
    
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_02')
    print(new_file_name)

    # aggiunta di righe per i nuovi parametri
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)

    # aggiornamento metadati nel support file
    new_support_file = dataCleaner.update_metadati_support(new_support_file)
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
   
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)

    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name)   




 ---- ADNIMERGE_25Jul2025_01.csv
ADNIMERGE_25Jul2025_02.csv


 ---- MMSE_25Jul2025_01.csv
MMSE_25Jul2025_02.csv


 ---- PTDEMOG_25Jul2025_01.csv
PTDEMOG_25Jul2025_02.csv


 ---- ADSP_PHC_BIOMARKER_25Jul2025_01.csv
ADSP_PHC_BIOMARKER_25Jul2025_02.csv


 ---- BLCHANGE_25Jul2025_01.csv
BLCHANGE_25Jul2025_02.csv


 ---- DXSUM_25Jul2025_01.csv
DXSUM_25Jul2025_02.csv


In [9]:
search = client.query_files(
    query={'custom.level' : 'cleaned_02', 'custom.source' : 'ADNI'})
files = [x['filename'] for x in search['included_files']]
print(files)

['ADNIMERGE_25Jul2025_02.csv', 'MMSE_25Jul2025_02.csv', 'PTDEMOG_25Jul2025_02.csv', 'ADSP_PHC_BIOMARKER_25Jul2025_02.csv', 'BLCHANGE_25Jul2025_02.csv', 'DXSUM_25Jul2025_02.csv']
